In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb

from sklearn.metrics import classification_report #accuracy_score, precision_score, recall_score, 
from sklearn.model_selection import train_test_split

In [ ]:
seed = 42
y_col = "Cover_Type"

## 1. データの読み込み

In [ ]:
df_data = pd.read_csv("../data/data.csv")

## 2. データ量とモデルの精度の実験

In [ ]:
def train_predict_eval(
    X_train_splitted, y_train_splitted, 
    X_val, y_val, 
    X_test, y_test,
    percentage_split
):
    
    model = lgb.LGBMClassifier(
        objective="multiclass",
        num_class=y_train_splitted.nunique(),
        class_weight="balanced",
        random_state=seed,
        verbose=-1,        # ログを非表示
        n_estimators=1000  # Early Stoppingを使うため大きめの数値を設定
    )

    print("train starts")
    model.fit(
        X_train_splitted, y_train_splitted,
        eval_set=(X_val, y_val),
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )

    print("prediction starts")
    y_pred = model.predict(X_test)

    print("pred ends\n")
    print(f"percentage_split(ratio to all):{percentage_split}[%]\n",
            classification_report(y_test, y_pred))
    print(X_train_splitted.shape)
    print(y_train_splitted.value_counts())

In [ ]:
X = df_data.drop(y_col, axis=1).copy()
y = df_data[y_col].copy()

In [ ]:
# データセットを指定した割合で分割する

# テストデータを全体の20%にする
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=seed
)

# 検証データを全体の20%にする
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, stratify=y_train_val, random_state=seed
)


In [ ]:
# 学習データの一部を分割する
# 学習データが全体の60％なので、(percentage_split/60)を指定することで、全体のpercentage_splitにする

percentage_split = 0.5

_, X_train_splitted, _, y_train_splitted = train_test_split(
    X_train, y_train, test_size=(percentage_split/60), stratify=y_train, random_state=seed
)

train_predict_eval(
    X_train_splitted=X_train_splitted, y_train_splitted=y_train_splitted, 
    X_val=X_val, y_val=y_val, 
    X_test=X_test, y_test=y_test,
    percentage_split=percentage_split
)
